# Geladen massa: Reissner-Nordström

In de vorige notebooks beschreven we de zwaartekracht van een ongeladen massa.
In dit notebook voegen we elektrische lading toe en onderzoeken het effect
op de ruimtetijdgeometrie.

In [ ]:
import sys, pathlib
sys.path.insert(0, str(pathlib.Path().resolve().parent / 'shared'))
from ort_core import (C, G, K_E, GravityModel, M_SUN, R_SUN)
import math
import numpy as np
import matplotlib.pyplot as plt
%matplotlib inline

---
# 1 Lading als energiebijdrage

## 1.1 Twee energietermen

In de ORT komt de zwaartekracht van een ongeladen massa uit de
gravitatie-energieparameter $v_{grav}^2 = 2GM/r$. Een geladen massa
heeft een extra energiebijdrage: het elektrische veld zelf bevat energie.

De elektrostatische veldenergie buiten straal $r$ is $E_{elec} = k_e Q^2/(2r)$ —
de zelfenergie van het elektrische veld van de centrale massa. Deze energie
is altijd positief (proportioneel met $Q^2$) en verlaagt de effectieve
valsnelheid, ongeacht de lading van een testdeeltje.

Omgerekend naar een energieparameter:
$v_{lading}^2 = 2 E_{elec} \cdot G / (c^2 r) = k_e Q^2 G/(c^2 r^2)$.

De effectieve valsnelheid:

$$v_{eff}^2 = v_{grav}^2 - v_{lading}^2 = \frac{2GM}{r} - \frac{k_e Q^2 G}{c^2 r^2} = c^2\left(\frac{r_s}{r} - \frac{r_Q^2}{r^2}\right) \qquad (1)$$

met $r_Q^2 = k_e Q^2 G / c^4$.

## 1.2 De metriekfunctie

Uit de snelheidsnorm $c_{local}^2 = c^2 - v_{eff}^2$ volgt:

$$c_{local}(r) = c \cdot \sqrt{1 - \frac{r_s}{r} + \frac{r_Q^2}{r^2}} \qquad (2)$$

## 1.3 Vergelijking met de ART

De ART leidt de Reissner-Nordström-metriekfunctie af als oplossing van de
Einstein-vergelijkingen met een elektromagnetische brontensor:

$$f(r) = \left(\frac{c_{local}}{c}\right)^2 = 1 - \frac{r_s}{r} + \frac{r_Q^2}{r^2} \qquad (3)$$

De ORT komt tot dezelfde formule vanuit twee energiebijdragen aan de
snelheidsnorm: gravitatie (trekt) en elektrostatisch veld (duwt terug).

## 1.2 Twee horizonten

De eventhorizon ligt waar $f(r) = 0$. Voor een geladen massa zijn er
**twee** oplossingen:

$$r_{\pm} = \frac{r_s \pm \sqrt{r_s^2 - 4r_Q^2}}{2} \qquad (3)$$

- $r_+$: de **buitenste horizon** (analoog aan de Schwarzschild-horizon)
- $r_-$: de **binnenste horizon** (Cauchy-horizon)

Tussen de twee horizonten is $f(r) < 0$ — de ruimte en tijd wisselen
van karakter, net als binnen een Schwarzschild-horizon.

### Extremale lading

Wanneer $r_+ = r_-$ (de discriminant is nul):

$$Q_{ext} = \frac{c^2}{2} \sqrt{\frac{r_s^2}{k_e G}} \qquad (4)$$

Bij $Q > Q_{ext}$ verdwijnen beide horizonten — een **naakte singulariteit**.
Het kosmische censuurvermoeden stelt dat dit in de natuur niet voorkomt.

In [ ]:
# Reissner-Nordström: vergelijking Schwarzschild vs geladen
M_BH = 10 * M_SUN
bh_sch = GravityModel(M_BH)
Q_ext = bh_sch.extremal_charge()
bh_half = GravityModel(M_BH, charge=0.5 * Q_ext)
bh_ext = GravityModel(M_BH, charge=0.99 * Q_ext)

print(f'=== Reissner-Nordström (10 M☉) ===')
print(f'r_s = {bh_sch.rs:.3e} m')
print(f'Q_ext = {Q_ext:.3e} C')
print()

for label, model in [('Q = 0 (Schwarzschild)', bh_sch),
                      ('Q = 0.5 Q_ext', bh_half),
                      ('Q = 0.99 Q_ext', bh_ext)]:
    r_plus = model.r_plus
    r_minus = model.r_minus
    print(f'--- {label} ---')
    print(f'  r+ = {r_plus:.3e} m = {r_plus/bh_sch.rs:.4f} r_s')
    print(f'  r- = {r_minus:.3e} m = {r_minus/bh_sch.rs:.4f} r_s')
    b = 10 * bh_sch.rs
    alpha = model.light_deflection_arcsec(b)
    print(f'  Lichtafbuiging (b=10r_s): {alpha:.6f}"')
    print()

---
# 2 Effecten van lading

## 2.1 Lading werkt zwaartekracht tegen

De ladingsterm verhoogt $c_{local}$, wat alle gravitatie-effecten verzwakt:

| Effect | Ongeladen | Geladen | Verandering |
|--------|-----------|---------|-------------|
| Tijddilatatie | sterker | zwakker | minder vertraging |
| Lichtafbuiging | groter | kleiner | minder afbuiging |
| Baanprecissie | groter | kleiner | minder precessie |
| Eventhorizon | $r_s$ | $r_+ < r_s$ | horizon krimpt |

Fysisch: de elektrostatische energie in het veld heeft een
"afstotend" effect op de ruimtetijdgeometrie.

In [ ]:
# c_local profielen: Schwarzschild vs RN
r_over_rs = np.linspace(1.01, 10, 500)

fig, ax = plt.subplots(figsize=(10, 6))

for label, model, color, ls in [
    ('Q = 0 (Schwarzschild)', bh_sch, 'black', '-'),
    ('Q = 0.5 Q_ext', bh_half, 'blue', '--'),
    ('Q = 0.99 Q_ext', bh_ext, 'red', '-.')]:
    cl = [model.c_local(f * bh_sch.rs) / C for f in r_over_rs]
    ax.plot(r_over_rs, cl, color=color, ls=ls, lw=2, label=label)

ax.set_xlabel('$r / r_s$', fontsize=12)
ax.set_ylabel('$c_{local} / c$', fontsize=12)
ax.set_title('$c_{local}$ profiel: effect van lading', fontsize=14, fontweight='bold')
ax.legend(fontsize=11)
ax.grid(True, alpha=0.3)
ax.set_xlim(1, 10)
ax.set_ylim(0, 1.05)
plt.tight_layout()
plt.show()

## 2.2 De horizonstructuur

De twee horizonten als functie van de lading:

In [ ]:
# Horizonten als functie van Q/Q_ext
q_frac = np.linspace(0, 1, 200)
r_plus_arr = []
r_minus_arr = []

for qf in q_frac:
    model = GravityModel(M_BH, charge=qf * Q_ext)
    r_plus_arr.append(model.r_plus / bh_sch.rs)
    r_minus_arr.append(model.r_minus / bh_sch.rs)

fig, ax = plt.subplots(figsize=(10, 6))
ax.plot(q_frac, r_plus_arr, 'b-', lw=2, label='$r_+$ (buitenste horizon)')
ax.plot(q_frac, r_minus_arr, 'r--', lw=2, label='$r_-$ (binnenste horizon)')
ax.fill_between(q_frac, r_minus_arr, r_plus_arr, alpha=0.1, color='purple')
ax.set_xlabel('$Q / Q_{ext}$', fontsize=12)
ax.set_ylabel('$r / r_s$', fontsize=12)
ax.set_title('Horizonten als functie van lading', fontsize=14, fontweight='bold')
ax.legend(fontsize=11)
ax.grid(True, alpha=0.3)
ax.axhline(0.5, color='gray', ls=':', lw=1, label='$r_s/2$')
ax.set_xlim(0, 1)
ax.set_ylim(0, 1.1)
plt.tight_layout()
plt.show()

print(f'Bij Q = Q_ext: r+ = r- = {bh_sch.rs/2:.3e} m = 0.5 r_s')

---
# 3 Vergelijking met Schwarzschild

De effecten van lading op de drie klassieke testen:

In [ ]:
# Vergelijking: lichtafbuiging en precissie voor verschillende ladingen
b = 10 * bh_sch.rs
a_orbit = 50 * bh_sch.rs
e_orbit = 0.3

print(f'=== Effect van lading op waarneembare effecten ===')
print(f'Impactparameter b = 10 r_s, baanparameters a = 50 r_s, e = {e_orbit}')
print()
print(f'{"Q/Q_ext":>8s}  {"r+/r_s":>8s}  {"lichtafb.":>12s}  {"precissie/omloop":>18s}  {"c_local(5rs)":>14s}')
print('-' * 68)

for qf in [0.0, 0.1, 0.3, 0.5, 0.7, 0.9, 0.99]:
    model = GravityModel(M_BH, charge=qf * Q_ext)
    alpha = model.light_deflection_arcsec(b)
    dphi = model.orbital_precession(a_orbit, e_orbit)
    cl = model.c_local(5 * bh_sch.rs) / C
    print(f'{qf:8.2f}  {model.r_plus/bh_sch.rs:8.4f}  {alpha:12.6f}"  {math.degrees(dphi)*3600:18.6f}"  {cl:14.6f}')

---
# 4 ORT-interpretatie

In de ORT is de metriekfunctie $f(r) = (c_{local}/c)^2$. De lading
voegt een positieve term toe:

$$\frac{c_{local}^2}{c^2} = 1 - \frac{r_s}{r} + \frac{r_Q^2}{r^2}$$

De valsnelheid wordt effectief verlaagd:

$$v_{grav,eff}^2 = \frac{2GM}{r} - \frac{k_e Q^2 G}{c^2 r^2} = c^2\left(\frac{r_s}{r} - \frac{r_Q^2}{r^2}\right) \qquad (5)$$

De elektrostatische energie "duwt terug" tegen de gravitatie-energie.
Bij de extremale lading is $v_{grav,eff} = 0$ bij $r = r_s/2$ — de twee
horizonten vallen samen en de horizon is marginaal.

## Het riviermodel met lading

In het riviermodel stroomt de ruimte naar de massa toe met $v_{grav}$.
Lading vertraagt de stroom — alsof er een tegendruk is. Bij de extremale
lading is de stroom precies in balans: de ruimte stroomt nog net, maar
de horizon is een enkel punt.

---
# 5 Samenvatting

| | Schwarzschild | Reissner-Nordström |
|---|---|---|
| Metriekfunctie | $1 - r_s/r$ | $1 - r_s/r + r_Q^2/r^2$ |
| Horizonten | 1 ($r = r_s$) | 2 ($r_+$ en $r_-$) |
| Extremaal | — | $r_+ = r_- = r_s/2$ |
| $c_{local}$ | $c\sqrt{1 - r_s/r}$ | $c\sqrt{1 - r_s/r + r_Q^2/r^2}$ |
| Lading effect | — | Verzwakt alle gravitatie-effecten |
| ORT-interpretatie | Rivierstroming $v_{grav}$ | Vertraagde stroom door tegendruk |